# 00 Setup, Keys & Providers

Install dependencies, configure providers, and verify your environment before live agent runs.


In [ ]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Pick up edits to shared/ without a full kernel restart (re-run this cell)
for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


## Learning objectives

- Create and activate a Python 3.10+ virtual environment
- Configure `.env` without committing secrets
- Understand default provider wiring in `shared/llm.py`
- Know which notebooks require live API keys


In [ ]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


In [ ]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


## 1. Install stack

From repo root:
```bash
pip install -r requirements.txt
python -m ipykernel install --user --name langchain-bootcamp
```


## 2. Copy environment template

```bash
cp .env.example .env
```

Minimum: `DEEPSEEK_API_KEY` with `LLM_PROVIDER=deepseek`. Optional: `TAVILY_API_KEY`, `LANGSMITH_API_KEY`.


## Verify .env file

The bootcamp loads secrets from `.env` at the repo root, never from notebook cells.


Run the demo below to verify the behavior.


In [ ]:
from pathlib import Path
env_path = ROOT / ".env"
print("ROOT:", ROOT)
print("Found .env:", env_path.exists())
if not env_path.exists():
    print("ACTION: Copy .env.example to .env before live cells")


**Reflect:** If `Found .env: False`, live cells will fail, fix this first.


> **Requires DeepSeek**, set `DEEPSEEK_API_KEY` and `LLM_PROVIDER=deepseek` in `.env`. The next cell calls the live model.


## Test LLM connection

Confirms your provider wiring in `shared/llm.py` works.


Run the demo below to verify the behavior.


In [ ]:
model = require_llm()
print("LLM OK:", type(model).__name__)


**Reflect:** If the test fails, confirm `DEEPSEEK_API_KEY` is set in `.env` before notebook 02.


## Active provider

Shows which backend `require_llm()` will use.


Run the demo below to verify the behavior.


In [ ]:
import os
print("Active provider:", os.getenv("LLM_PROVIDER", "deepseek"))
print("Keys set:", {
    k: bool(os.getenv(k))
    for k in ["DEEPSEEK_API_KEY", "TAVILY_API_KEY", "LANGSMITH_API_KEY"]
})


## Production note

Use separate API keys per environment (dev/staging/prod). Rotate keys if ever exposed in chat logs or notebook outputs.


Run the demo below to verify the behavior.


## Common mistakes

- Committing `.env`, it must stay gitignored
- Pasting keys into notebook cells, use environment variables only
- Using different embedding models for index vs query
